In [34]:
# library imports

import pandas as pd
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [35]:
# Downloading the required data for testing

# Not downloading this every time :)
# nltk.download('all')

# Gonna start testing how this works with the Amazon reviews dataset.
df = pd.read_csv('https://raw.githubusercontent.com/pycaret/pycaret/master/datasets/amazon.csv')
df.head(5)

,reviewText,Positive
0,This is a one of the best apps acording to a b...,1
1,This is a pretty good version of the game for ...,1
2,this is a really cool game. there are a bunch ...,1
3,"This is a silly game and can be frustrating, b...",1
4,This is a terrific game on any pad. Hrs of fun...,1


In [49]:
# Defining the preprocessing for text
def preprocess_text(text):

    # Tokenising:
    tokens = word_tokenize(str(text).lower())

    # Remove stop words, i.e. words that generally don't add any emotion to sentences, e.g. and, but, etc.

    filtered_tokens = [token for token in tokens if (token not in stopwords.words('english') or token not in "0123456789")]

    # Lemmatize the tokens, i.e. converting all words into their base form. For example, "gaming" -> game, "hiking" -> hike, and so on.

    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in filtered_tokens]

    # Merging all the tokens back into one string

    processed_text = ' '.join(lemmatized_tokens)

    return processed_text

# Applying the function to our data.

df['reviewText'] = df['reviewText'].apply(preprocess_text)
df.head(5)

,reviewText,Positive,sentiment
0,this is a one of the best apps acording to a b...,1,1
1,this is a pretty good version of the game for ...,1,1
2,this is a really cool game . there are a bunch...,1,1
3,"this is a silly game and can be frustrating , ...",1,1
4,this is a terrific game on any pad . hr of fun...,1,1


In [50]:
# Creating the NLTK sentiment analyser.

analyzer = SentimentIntensityAnalyzer()


# Defining the sentiment retrieval function which just utilises the NLTK library 
# to output the overall sentiment for our text input parameter.

def get_sentiment(text):

    scores = analyzer.polarity_scores(text)

    sentiment = 1 if scores['pos'] > 0 else 0

    return sentiment

# Applying the same function to our dataset.

df['sentiment'] = df['reviewText'].apply(get_sentiment)
df.head(5)

,reviewText,Positive,sentiment
0,this is a one of the best apps acording to a b...,1,1
1,this is a pretty good version of the game for ...,1,1
2,this is a really cool game . there are a bunch...,1,1
3,"this is a silly game and can be frustrating , ...",1,1
4,this is a terrific game on any pad . hr of fun...,1,1


In [51]:
# Classic confusion matrix since we have a categorisation task at hand.

print(confusion_matrix(df['Positive'], df['sentiment']))

[[ 1238  3529]
 [  510 14723]]


In [52]:
# For my own learning, this report outputs precision, recall, F1 score, and support.
# Precision: Of all the items the model predicted as positive, how many were actually positive?
# Recall: Of all the actual positive items, how many did the model correctly find? 
# F1 Score: A mix of precision and recall, used when we want to consider both false negatives and false positives in our review of the model.
# Support: How many true examples there are for each class in your ground truth labels.

print(classification_report(df['Positive'], df['sentiment']))

              precision    recall  f1-score   support

           0       0.71      0.26      0.38      4767
           1       0.81      0.97      0.88     15233

    accuracy                           0.80     20000
   macro avg       0.76      0.61      0.63     20000
weighted avg       0.78      0.80      0.76     20000



In [53]:
# Now going to use my friends' messages.

disc_df = pd.read_csv("messages.csv")
disc_df.head(5)

,timestamp,guild,channel,author,content
0,2018-12-10 17:41:16.617000+00:00,onyx worshipping group,announcements,FUCK KENNY'S DEAD BODY,NaN
1,2018-12-11 17:06:04.690000+00:00,onyx worshipping group,announcements,o wait,NaN
2,2018-12-11 17:06:09.770000+00:00,onyx worshipping group,announcements,im an idiot,NaN
3,2018-12-11 17:06:30.713000+00:00,onyx worshipping group,announcements,@everyone if you were given all the money you ...,NaN
4,2018-12-11 17:29:09.331000+00:00,onyx worshipping group,announcements,https://www.reddit.com/r/Animemes/comments/a56...,NaN


In [68]:
# Just wanting the message column from the dataset, and renaming it a better column name since the data scraped from
# the Discord API resulted in this inaccuracy in naming.
msg_df = disc_df[['author']].copy().rename(columns={'author':'message'})
msg_df.head(5)

,message
0,FUCK KENNY'S DEAD BODY
1,o wait
2,im an idiot
3,@everyone if you were given all the money you ...
4,https://www.reddit.com/r/Animemes/comments/a56...


In [69]:
# Now utilising all of the methods seen above for the sample dataset on our actual dataset.

msg_df['message'] = msg_df['message'].apply(preprocess_text)
msg_df.head(5)

,message
0,fuck kenny 's dead body
1,o wait
2,im an idiot
3,@ everyone if you were given all the money you...
4,http : //www.reddit.com/r/animemes/comments/a5...


In [70]:
msg_df['sentiment'] = msg_df['message'].apply(get_sentiment)
msg_df.head(5)

,message,sentiment
0,fuck kenny 's dead body,0
1,o wait,0
2,im an idiot,0
3,@ everyone if you were given all the money you...,0
4,http : //www.reddit.com/r/animemes/comments/a5...,0


In [77]:
pos_neg_dico = {"pos":0, "neg":0}
for rec in msg_df['sentiment']:
    if int(rec) == 0:
        pos_neg_dico["neg"] += 1
    else:
        pos_neg_dico["pos"] += 1

disc_pos_score = round((pos_neg_dico["pos"] / (pos_neg_dico["pos"] + pos_neg_dico["neg"])) * 100, 2)

print(f"Your friendship group's positivity score is {disc_pos_score}%!")

Your friendship group's positivity score is 19.14%!
